# Moirai VaR-aware Training on Kaggle
Notebook này được tạo tự động để chạy full-pipeline huấn luyện mô hình Moirai VaR-aware.

**Yêu cầu môi trường:**
- Dataset: Đã được mount tại `/kaggle/input/datasets/trnhngv/historical-price`
- Weights: Đã được mount tại `/kaggle/input/datasets/trnhngv/weights`
- Tùy chọn GPU: Hãy bật T4 x2 hoặc P100 trên Kaggle.

In [ ]:
# 1. Setup Môi trường: Clone Source Code (Thay <URL_REPO_CỦA_BẠN> bằng link GitHub thật)
!git clone -b statistitcal-interpretation <URL_REPO_CỦA_BẠN> repo

# Cài đặt các thư viện cần thiết
!pip install gluonts torch pandas numpy scipy jaxtyping "jax[cpu]"

In [ ]:
# 2. Cấu hình Python Path
import sys
import os

# Đưa thư mục repo vào sys.path để import được các module trong experiments/
sys.path.insert(0, '/kaggle/working/repo')

# Đưa thư mục uni2ts (có sẵn trong repo) vào sys.path
sys.path.insert(0, '/kaggle/working/repo/model/Morai based/uni2ts/src')

In [ ]:
# 3. Import các thư viện và hàm cốt lõi
import torch
import pandas as pd
from torch.utils.data import DataLoader

from experiments.moirai_var_aware.config import SPLIT_INFO
from experiments.moirai_var_aware.data import load_and_split_dataset
from experiments.moirai_var_aware.export import build_prediction_frame
from experiments.moirai_var_aware.modeling import (
    VolatilityFeatureExtractor,
    VolatilityRegressionModel,
    patch_uni2ts_exports,
)
from experiments.moirai_var_aware.train import predict, train_model_var_aware

# Patch thư viện uni2ts với đường dẫn cụ thể trên Kaggle
patch_uni2ts_exports(base='/kaggle/working/repo/model/Morai based/uni2ts/src/uni2ts/model')

In [ ]:
# 4. Cấu hình Tham số Huấn luyện
DATASET_DIR = '/kaggle/input/datasets/trnhngv/historical-price'
WEIGHTS_DIR = '/kaggle/input/datasets/trnhngv/weights'

MODELS_TO_TRAIN = ["moirai", "moirai2", "moirai_moe"]
DATASETS_TO_TRAIN = list(SPLIT_INFO.keys())  # Bạn có thể chỉ định riêng mảng, vd: ["VN_INDEX"]

LAMBDA_VAR = 0.2
EPOCHS = 30
BATCH_SIZE = 8  # Chỉnh nhỏ lại (vd: 4 hoặc 8) nếu bị lỗi OOM (Out of Memory) trên Kaggle
TUNING_MODE = "head"  # Các tuỳ chọn: "head", "full"
BACKBONE_LR = 1e-5

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Đang sử dụng thiết bị:", device)

In [ ]:
# 5. Vòng lặp Training (Full Pipeline)
all_frames = []
csv_outputs = {} # Lưu trữ nội dung CSV gốc theo tên file

for index_name in DATASETS_TO_TRAIN:
    csv_path = os.path.join(DATASET_DIR, f"{index_name}.csv")
    if not os.path.exists(csv_path):
        print(f"Bỏ qua {index_name} vì không tìm thấy file: {csv_path}")
        continue

    print(f"\n========== Đang xử lý Dataset: {index_name} ==========")
    train_ds, val_ds, test_ds = load_and_split_dataset(csv_path, index_name)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False)

    for model_type in MODELS_TO_TRAIN:
        print(f"\n--- {index_name} | Model: {model_type} | Lambda: {LAMBDA_VAR} | Mode: {TUNING_MODE} ---")
        
        extractor = VolatilityFeatureExtractor(
            model_type=model_type,
            size="small",
            device=device,
            weights_dir=WEIGHTS_DIR,
            freeze_backbone=(TUNING_MODE == "head"),
        )
        model = VolatilityRegressionModel(extractor=extractor)
        
        # Bắt đầu quá trình học
        model = train_model_var_aware(
            model,
            train_loader,
            val_loader,
            epochs=EPOCHS,
            lr=1e-3,
            backbone_lr=BACKBONE_LR,
            device=device,
            alpha=0.01,
            lambda_var=LAMBDA_VAR,
            distribution="student_t",
            nu=4.0,
            var_horizon_index=0,
            tuning_mode=TUNING_MODE,
        )
        
        # Dự đoán trên tập Test
        preds, targets = predict(model, test_loader, device=device)
        
        # Lấy kết quả trả về từ hàm build_prediction_frame
        frame = build_prediction_frame(
            index_name, model_type, LAMBDA_VAR, test_ds, preds, targets, tuning_mode=TUNING_MODE
        )
        all_frames.append(frame)
        
        # Định dạng tên file theo đúng chuẩn của file MD gốc
        mode_suffix = "" if TUNING_MODE == "head" else f"_{TUNING_MODE}"
        filename = f"{index_name}_{model_type}{mode_suffix}_lambda_{LAMBDA_VAR:g}_predictions.csv"
        csv_outputs[filename] = frame.to_csv(index=False)
        
        # Giải phóng bộ nhớ RAM/VRAM để train model tiếp theo
        del model
        del extractor
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
# 6. Lưu kết quả gốc thành file Zip (Output chuẩn theo document)
import zipfile

if len(csv_outputs) > 0:
    mode_str = "" if TUNING_MODE == "head" else "_full"
    zip_filename = f"/kaggle/working/moirai_var{mode_str}_lambda_{LAMBDA_VAR:g}_predictions.zip"
    
    with zipfile.ZipFile(zip_filename, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for filename, content in csv_outputs.items():
            zf.writestr(filename, content)
    print(f"✅ Đã tạo file Zip nguyên bản tại: {zip_filename} (Chứa {len(csv_outputs)} file csv)")
else:
    print("Không có kết quả gốc nào để nén Zip.")